In [4]:
%pip install stanza

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 129.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 17.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
No

In [ ]:
#from very start, when trying to use paraphrase encodings

from torch import nn
import torch
import pandas as pd
import stanza
from collections import Counter
from torch.optim.lr_scheduler import OneCycleLR

In [6]:
stanza.download('en')
nlp = stanza.Pipeline(lang='en', processors='tokenize,pos,constituency')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

2026-06-19 14:09:16 INFO: Downloaded file to /home/klw66/.cache/stanza/1.13.0/resources/resources.json
2026-06-19 14:09:16 INFO: Downloading default packages for language: en (English) ...


models/default.zip:   0%|          | 0.00/525M [00:00<?, ?B/s]

2026-06-19 14:09:23 INFO: Downloaded file to /home/klw66/.cache/stanza/1.13.0/resources/en/default.zip
2026-06-19 14:09:26 INFO: Finished downloading models and saved to /home/klw66/.cache/stanza/1.13.0/resources
2026-06-19 14:09:26 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2026-06-19 14:09:27 INFO: Downloaded file to /home/klw66/.cache/stanza/1.13.0/resources/resources.json
2026-06-19 14:09:27 WARNING: Language en package default expects mwt, which has been added
2026-06-19 14:09:27 INFO: Loading these models for language: en (English):
| Processor    | Package             |
--------------------------------------
| tokenize     | combined            |
| mwt          | combined            |
| pos          | combined_charlm     |
| constituency | ptb3-revised_charlm |

2026-06-19 14:09:27 INFO: Using device: cuda
2026-06-19 14:09:27 INFO: Loading: tokenize
2026-06-19 14:09:31 INFO: Loading: mwt
2026-06-19 14:09:31 INFO: Loading: pos
2026-06-19 14:09:32 INFO: Loading: constituency
2026-06-19 14:09:32 INFO: Done loading processors!


In [7]:
def cast_to_float(context):
    return([float(x) for x in context.strip("[]").split()])
# def clean_text(text):
#     if pd.isna(text):
#         return text

#     # Remove single quotes around a word/phrase, e.g. 'Oliver' -> Oliver
#     text = re.sub(r"'([\w\s\-]+)'", r'\1', text)

#     # Replace & with "n" in abbreviations like B&Q -> BnQ
#     # Matches a letter/digit, then &, then a letter/digit (no spaces around &)
#     text = re.sub(r'(?<=\w)&(?=\w)', 'n', text)

#     # Lowercase everything
#     text = text.lower()

#     # Add a space before and after every forward slash
#     text = re.sub(r'\s*/\s*', ' / ', text)

#     # Add a space before and after every hyphen
#     text = re.sub(r'\s*-\s*', ' - ', text)

#     # Collapse repeated ! or ? into a single one, e.g. "!!!" -> "!", "???" -> "?"
#     text = re.sub(r'!+', '!', text)
#     text = re.sub(r'\?+', '?', text)

#     return text

In [9]:
df = pd.read_csv("paraphrase_encodings.csv", engine="python")
# df = df.iloc[:-49_000]
df['context_vector'] = df['context_vector'].apply(lambda x: cast_to_float(x))
# df['source'] = df['source'].apply(clean_text)

ParserError: unexpected end of data

In [206]:
model_size = len(df['context_vector'].iloc[0])
print(model_size)

384


In [ ]:
# all_tokens = []
# vocab = {}
# for sentence in (df['source'].tolist()):
#     tokens = word_tokenize(sentence)
#     all_tokens.extend(tokens)
# for token, _ in Counter(all_tokens).most_common():
#     vocab[token] = len(vocab)

# vocab_size = len(vocab)
# print(vocab_size)

37144


In [219]:
def skip_unary(node):
    """Follow unary chains down until reaching a node with 0 or 2+ children."""
    while len(node.children) == 1:
        node = node.children[0]
    return node

vocab = {}
def stanza_tree_to_nested(node, role=0, max_role = None):
    """
    Recursively convert a Stanza constituency tree node to a nested list,
    collapsing unary chains.

    Each leaf becomes:        [word, role]
    Each inner node becomes:  [[child_0, child_1, ...], role]
    where each child's own role is its index among its siblings.
    Unary chains (nodes with a single child, recursively) are skipped.

    Args:
        node: a Stanza constituency tree node
        role: the index of this node among its parent's children (default 0 for root)

    Returns:
        A nested list representation of the subtree.
    """
    if max_role is None:
        max_role = [0]  # mutable container so mutations propagate up

    # Collapse unary chains before processing
    node = skip_unary(node)

    if role > max_role[0]:
        max_role[0] = role

    # Leaf node: return [word, role]
    if len(node.children) == 0:
        label = node.label.lower()
        try:
            token = torch.tensor(vocab[label], dtype=torch.long)
        except KeyError:
            token = torch.tensor(len(vocab), dtype=torch.long)
            vocab[label] = len(vocab)
        return [token, torch.tensor(role, dtype = torch.long)], max_role[0]

    # Inner node: children go into their own sublist, then role is appended
    converted_children = [
        stanza_tree_to_nested(child, sibling_index, max_role)[0]
        for sibling_index, child in enumerate(node.children)
    ]
    return [converted_children, torch.tensor(role, dtype = torch.long)], max_role[0]

In [220]:
all_src = []
glob_max_role = 1
for i in df["source"].tolist():
    doc = nlp(i.lower())
    tree, max_role = stanza_tree_to_nested(doc.sentences[0].constituency)
    if max_role > glob_max_role:
        glob_max_role = max_role
    if isinstance(tree[0], list):
        all_src.append(tree[0])
    else:
        all_src.append([tree])


In [218]:
test = "Minnesota Wild reassign four players to Aeros"
doc = nlp(test.lower())
tree, max_role = stanza_tree_to_nested(doc.sentences[0].constituency)
print(tree)

[[[[[tensor(1434), tensor(0)], [tensor(714), tensor(1)]], tensor(0)], [[[tensor(3639), tensor(0)], [[[tensor(2122), tensor(0)], [tensor(839), tensor(1)]], tensor(1)], [[[tensor(50), tensor(0)], [tensor(3640), tensor(1)]], tensor(2)]], tensor(1)]], tensor(0)]


In [221]:
vocab_size = len(vocab)
print(vocab_size)

3649


In [222]:
from sklearn.model_selection import train_test_split


all_trg = df["context_vector"].tolist()
# random_state=42
# First split: 80% train, 20% temp (which becomes val+test)
train_src, temp_src, train_trg, temp_trg = train_test_split(
    all_src, all_trg, test_size=0.2
)

# Second split: split the 20% temp into 10% val, 10% test
val_src, test_src, val_trg, test_trg = train_test_split(
    temp_src, temp_trg, test_size=0.5
)

print(len(train_src), len(val_src), len(test_src))

800 100 100


In [223]:
class RTPREncode(nn.Module):
    def __init__(self, nfillers, nroles, filler_size, role_size, model_size):
        super(RTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_size)
        self.roles = nn.Embedding(nroles, role_size)
        self.filler_size = filler_size
        self.role_size = role_size
        self.model_size = model_size
        # self.dropout = nn.Dropout(0.4)
        self.w = torch.nn.Parameter(torch.randn(filler_size, filler_size*role_size)*0.01)
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_size)*0.01)

    def forward(self, x):
        result = torch.zeros(self.filler_size, self.role_size, device=self.w.device)
        for pair in x:
            # print(pair)
            filler = pair[0]
            role = pair[1].to(device)
            
            if isinstance(filler, list):
                filler = self(filler)
            else:
                filler = self.fillers(filler.to(device))
            
            role = self.roles(role)
            result += torch.outer(filler, role)
        
        result = torch.flatten(result)
        result = self.w @ result
        return result

In [226]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, src, trg, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for i in range(len(src)):
            src_batch = src[i]
            trg_batch = torch.tensor(train_trg[i], dtype=torch.long).to(device)
            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)    # (batch_size, model_size)
            loss = criterion(output, trg_batch)
            epoch_loss += loss.item()

        return epoch_loss / len(src)

In [216]:
all_keys = [k for k, v in vocab.items() if v == 1434]
print(all_keys)

# print(vocab[12235])
print(df["source"].iloc[44])

['minnesota']
Fire guts labour accommodation in Ras Al Khaimah


In [227]:
# Train with optimized hyperparams
num_epochs = 50

model = RTPREncode(vocab_size, glob_max_role + 1, 8, 8, model_size).to(device)
learning_rate = 1e-4
norm_val = 5
max_rate = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    # print(f"Trial {trial.number}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
        for i in range(len(train_src)):
            src_batch = train_src[i]
            trg_batch = torch.tensor(train_trg[i], dtype=torch.long).to(device)
            optimizer.zero_grad()
            try:
                encoding = model(src_batch)
            except TypeError:
                print(src_batch)
                print(i)
            except IndexError:
                print(src_batch)
                print(i)
            output = torch.matmul(encoding, model.wo.T)

            loss = criterion(output, trg_batch)
            loss.backward()

            #Norm=magnitude, clipping gradient to ensure they don't explode
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

    print(f"Train Loss: {epoch_loss/len(train_src)}")
    val_loss = evaluate(model, val_src, val_trg, criterion, device)
    print(f"Validation Loss: {val_loss}")

Epoch 1 / 50
Train Loss: 0.0005260663202682158
Validation Loss: 0.0004434064492604328
Epoch 2 / 50
Train Loss: 0.0005250956048728582
Validation Loss: 0.0004428681520059785
Epoch 3 / 50
Train Loss: 0.000524435482480937
Validation Loss: 0.000442595949551432
Epoch 4 / 50
Train Loss: 0.000524090518276173
Validation Loss: 0.000442551883227767
Epoch 5 / 50
Train Loss: 0.0005239498527431155
Validation Loss: 0.0004425985228692443
Epoch 6 / 50
Train Loss: 0.0005238948982863831
Validation Loss: 0.00044265181171914455
Epoch 7 / 50
Train Loss: 0.0005238865932183255
Validation Loss: 0.0004426859618820923
Epoch 8 / 50
Train Loss: 0.0005239110911232823
Validation Loss: 0.00044270002160659496
Epoch 9 / 50
Train Loss: 0.0005239534216765571
Validation Loss: 0.00044270441771577157
Epoch 10 / 50
Train Loss: 0.0005239990379721507
Validation Loss: 0.00044270610608785135
Epoch 11 / 50
Train Loss: 0.0005240382642382317
Validation Loss: 0.00044270716031871834
Epoch 12 / 50
Train Loss: 0.0005240660285087477
Val

In [ ]:
def objective(trial):
    num_epochs = 50

    model = RTPREncode(vocab_size, max_role + 1, 8, 8, model_size).to(device)
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-3, log = True)
    norm_val = trial.suggest_float("grad_norm", 1, 10, log = True)
    max_rate = trial.suggest_float("scheduler_max", 1e-4, 1e-2, log = True)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1} / {num_epochs}")
        # print(f"Trial {trial.number}")
        model.train()

        epoch_loss = 0.0
        with torch.set_grad_enabled(True):
            for i in range(len(train_src)):
                src_batch = train_src[i]
                trg_batch = train_trg[i].to(device)
                optimizer.zero_grad()

                encoding = model(src_batch)
                output = torch.matmul(encoding, model.wo.T)

                loss = criterion(output, trg_batch)
                loss.backward()

                #Norm=magnitude, clipping gradient to ensure they don't explode
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
                optimizer.step()
                scheduler.step()
                epoch_loss += loss.item()

        print(f"Train Loss: {epoch_loss/len(train_src)}")
        val_loss = evaluate(model, val_src, val_trg, criterion, device)

        trial.report(val_loss, epoch)
        #prune cuts off trial early
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        print(f"Validation Loss: {val_loss}")

    return val_loss
    

[tensor(0), tensor(0)]
[[[tensor(1), tensor(0)], [[[tensor(2), tensor(0)], [tensor(3), tensor(1)]], tensor(1)]], tensor(1)]
[tensor(1), tensor(0)]
[[[tensor(2), tensor(0)], [tensor(3), tensor(1)]], tensor(1)]
[tensor(2), tensor(0)]
[tensor(3), tensor(1)]
torch.Size([24])
